# ML Zoomcamp 2026 - Homework 3: Machine Learning for Classification

Goal: predict `converted` (did the lead sign up to the platform or not). This is **binary classification**: the target is 0 or 1.

In [1]:
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.metrics import mutual_info_score, accuracy_score
from sklearn.feature_extraction import DictVectorizer
from sklearn.linear_model import LogisticRegression

## Load the data

In [2]:
url = 'https://raw.githubusercontent.com/DataTalksClub/machine-learning-zoomcamp/main/cohorts/2026/data/course_lead_scoring_2026.csv'
df = pd.read_csv(url)
print(df.shape)
df.head()

(5000, 9)


,lead_source,industry,employment_status,location,annual_income,number_of_courses_viewed,interaction_count,lead_score,converted
0,organic_search,technology,employed,europe,88160.0,3,4,0.64,1
1,social_media,technology,employed,south_america,72688.0,3,7,0.72,1
2,referral,retail,employed,europe,44697.0,3,4,0.58,1
3,paid_ads,manufacturing,student,NaN,NaN,3,6,0.57,0
4,referral,manufacturing,student,north_america,24062.0,4,5,0.62,1


In [3]:
df.dtypes

lead_source                     str
industry                        str
employment_status               str
location                        str
annual_income               float64
number_of_courses_viewed      int64
interaction_count             int64
lead_score                  float64
converted                     int64
dtype: object

## Data preparation

Split the columns into categorical (text) and numerical, then check for missing values.

In [4]:
categorical = ['lead_source', 'industry', 'employment_status', 'location']
numerical = ['number_of_courses_viewed', 'annual_income', 'interaction_count', 'lead_score']

df.isnull().sum()

lead_source                 148
industry                    240
employment_status           194
location                    208
annual_income               369
number_of_courses_viewed      0
interaction_count             0
lead_score                   35
converted                     0
dtype: int64

Fill the missing values as the homework says: `'NA'` for categorical features and `0.0` for numerical ones.

In [5]:
df[categorical] = df[categorical].fillna('NA')
df[numerical] = df[numerical].fillna(0.0)

df.isnull().sum().sum()

np.int64(0)

## Q1. Mode of `industry`

`value_counts()` shows every category with its frequency; the first one is the mode.

In [6]:
df['industry'].value_counts()

industry
technology       1173
retail            925
healthcare        840
finance           720
education         639
manufacturing     463
NA                240
Name: count, dtype: int64

In [7]:
df['industry'].mode()[0]

'technology'

## Q2. Correlation matrix of the numerical features

Correlation goes from -1 to 1. Close to 1: both features grow together; close to 0: no linear relation.

In [8]:
corr = df[numerical].corr()
corr.round(3)

,number_of_courses_viewed,annual_income,interaction_count,lead_score
number_of_courses_viewed,1.000,0.161,0.722,0.757
annual_income,0.161,1.000,0.123,0.229
interaction_count,0.722,0.123,1.000,0.916
lead_score,0.757,0.229,0.916,1.000


In [9]:
pairs = [
    ('interaction_count', 'lead_score'),
    ('number_of_courses_viewed', 'lead_score'),
    ('number_of_courses_viewed', 'interaction_count'),
    ('annual_income', 'interaction_count'),
]
for a, b in pairs:
    print(f'{a} - {b}: {corr.loc[a, b]:.4f}')

interaction_count - lead_score: 0.9157
number_of_courses_viewed - lead_score: 0.7572
number_of_courses_viewed - interaction_count: 0.7216
annual_income - interaction_count: 0.1228


## Split the data

60% train / 20% validation / 20% test, with the exact calls from the homework. The second split uses `test_size=0.25` because 25% of the remaining 80% is 20% of the total.

In [10]:
df_full_train, df_test = train_test_split(df, test_size=0.2, random_state=42)
df_train, df_val = train_test_split(df_full_train, test_size=0.25, random_state=42)

df_train = df_train.reset_index(drop=True)
df_val = df_val.reset_index(drop=True)
df_test = df_test.reset_index(drop=True)

y_train = df_train['converted'].values
y_val = df_val['converted'].values
y_test = df_test['converted'].values

# The target must not stay inside the feature dataframes
del df_train['converted']
del df_val['converted']
del df_test['converted']

len(df_train), len(df_val), len(df_test)

(3000, 1000, 1000)

## Q3. Mutual information

Mutual information measures how much knowing a categorical feature tells us about the target. Computed on the **training set only**.

In [11]:
mi = {col: round(mutual_info_score(df_train[col], y_train), 2) for col in categorical}
pd.Series(mi).sort_values(ascending=False)

lead_source          0.03
employment_status    0.02
industry             0.00
location             0.00
dtype: float64

## Q4. Logistic regression

`DictVectorizer` does the one-hot encoding: each category becomes its own 0/1 column, while numerical features pass through unchanged. It is fitted on train only, then just applied to validation.

In [12]:
def train_and_score(features, C=1.0):
    dv = DictVectorizer(sparse=False)
    X_train = dv.fit_transform(df_train[features].to_dict(orient='records'))
    X_val = dv.transform(df_val[features].to_dict(orient='records'))

    model = LogisticRegression(solver='liblinear', C=C, max_iter=1000, random_state=42)
    model.fit(X_train, y_train)

    y_pred = model.predict(X_val)
    return accuracy_score(y_val, y_pred)

In [13]:
all_features = categorical + numerical

base_accuracy = train_and_score(all_features)
print('accuracy:', base_accuracy)
round(base_accuracy, 2)

accuracy: 0.645


0.65

## Q5. Feature elimination

Remove one feature at a time and retrain. The difference is `original - without_feature` (not rounded). A difference close to 0 means the model barely needs that feature.

In [14]:
diffs = {}
for feature in all_features:
    subset = [f for f in all_features if f != feature]
    acc = train_and_score(subset)
    diffs[feature] = base_accuracy - acc
    print(f'{feature:28s} acc={acc:.4f}  diff={diffs[feature]:+.4f}')

lead_source                  acc=0.6420  diff=+0.0030
industry                     acc=0.6450  diff=+0.0000
employment_status            acc=0.6450  diff=+0.0000


location                     acc=0.6450  diff=+0.0000
number_of_courses_viewed     acc=0.6430  diff=+0.0020
annual_income                acc=0.7240  diff=-0.0790
interaction_count            acc=0.6010  diff=+0.0440
lead_score                   acc=0.6440  diff=+0.0010


In [15]:
candidates = ['lead_source', 'number_of_courses_viewed', 'interaction_count']
{f: round(diffs[f], 4) for f in candidates}

{'lead_source': 0.003,
 'number_of_courses_viewed': 0.002,
 'interaction_count': 0.044}

In [16]:
min(candidates, key=lambda f: diffs[f])

'number_of_courses_viewed'

## Q6. Regularization (parameter `C`)

In scikit-learn, **smaller `C` = stronger regularization** (the opposite of `r` in homework 2). Accuracy rounded to 3 digits; on a tie, choose the smallest `C`.

In [17]:
scores_c = {}
for C in [0.000001, 0.00001, 0.0001, 0.001]:
    scores_c[C] = round(train_and_score(all_features, C=C), 3)
    print(C, scores_c[C])

1e-06

 0.598
1e-05 0.598
0.0001 0.613
0.001 0.645


In [18]:
best = max(scores_c.values())
min(C for C, s in scores_c.items() if s == best)

0.001